In [2]:

import sys, os
sys.path.append(os.path.abspath("..")) 
import json
import torch
from datasets import CNFDataset
from models import LightningModelCNF
from utils import args_cnf, gen_image_cnf, plotly_generate

**Read configuration files and arguments:**

In [9]:
# Arguments
parser = args_cnf()
args, unknown = parser.parse_known_args()

args.particle = "proton_contained"
args.metadata_path = "/pscratch/sd/b/botaoli/SFGD_VA/Data/NN_Data_compressed/metadata.pkl"
args.dataset_path = "/pscratch/sd/b/botaoli/SFGD_VA/Data/NN_Data_compressed/{}/{}/{}/{}.zip"
args.cnf_ind_path = "/pscratch/sd/b/botaoli/SFGD_VA/Data/NN_Data_compressed/gan_ind.pkl"
args.save_dir = "/pscratch/sd/b/botaoli/SFGD_VA/Results/cnf/"
args.checkpoint_path = "/pscratch/sd/b/botaoli/SFGD_VA/Results/cnf/test_vanilla/checkpoints"
args.checkpoint_name = args.particle

if args.particle == "muon" or args.particle == "proton_exiting":
    args.label_size = 10
elif args.particle == "proton_contained":
    args.label_size = 7
args.epochs = 50
args.log_every_n_steps = 2000
args.batch_size = 512
args.hidden = 256
args.num_workers = 64

**Load the pre-trained weights of the different generative-adversarial-network (GAN) models:**

In [18]:
# Dataset and generator models
test_set_p = CNFDataset(args, split="val")




checkpoint_path = "/".join((args.checkpoint_path, args.checkpoint_name, "train_loss", "last.ckpt"))
# Load weights of pre-trained generator models
#checkpoint_p = torch.load(checkpoint_path, map_location='cpu')

# state_dict = {
#     key.replace("generator.", ""): value for key, value in checkpoint_p['state_dict'].items()
# }

# generator.load_state_dict(state_dict, strict=False)
# generator.eval();
#
# set the parameters of the model

model = LightningModelCNF.load_from_checkpoint(checkpoint_path, img_shape = (args.img_size, args.img_size, args.img_size), 
                                    label_size = args.label_size, 
                                    hidden_features = args.hidden, 
                                    num_blocks_in_MADE = args.num_blocks_in_MADE, 
                                    num_transformers = args.num_transformers, 
                                    lr = args.lr, 
                                    wd = args.weight_decay)

model.eval()

# move to cpu
model.to("cpu")


LightningModelCNF(
  (nflow): Flow(
    (_transform): CompositeTransform(
      (_transforms): ModuleList(
        (0-7): 8 x MaskedAffineAutoregressiveTransform(
          (autoregressive_net): MADE(
            (initial_layer): MaskedLinear(in_features=125, out_features=256, bias=True)
            (context_layer): Linear(in_features=7, out_features=256, bias=True)
            (activation): ReLU()
            (blocks): ModuleList(
              (0-1): 2 x MaskedFeedforwardBlock(
                (linear): MaskedLinear(in_features=256, out_features=256, bias=True)
                (activation): ReLU()
                (dropout): Dropout(p=0.0, inplace=False)
              )
            )
            (final_layer): MaskedLinear(in_features=256, out_features=250, bias=True)
          )
        )
        (8): RandomPermutation()
      )
    )
    (_distribution): StandardNormal()
    (_embedding_net): Identity()
  )
)

In [ ]:
print(checkpoint_p['state_dict'].keys())

In [4]:
print(test_set_p.metadata['statistics']['per_tree']['proton_contained']['recon_charge']['max'])
print(test_set_p.metadata['statistics']['per_tree']['proton_contained']['recon_charge']['min'])

2658
3


In [4]:
for key, value in checkpoint_p['state_dict'].items():
    print(key, value.shape)

generator.bert.cls_token torch.Size([1, 1, 64])
generator.bert.embedding.input.weight torch.Size([64, 1])
generator.bert.embedding.input.bias torch.Size([64])
generator.bert.embedding.label.embedding.weight torch.Size([64, 10])
generator.bert.embedding.label.embedding.bias torch.Size([64])
generator.bert.embedding.position.vol_idx torch.Size([126])
generator.bert.embedding.position.embedding.weight torch.Size([126, 64])
generator.bert.embedding.noise.weight torch.Size([64, 512])
generator.bert.embedding.noise.bias torch.Size([64])
generator.bert.transformer_blocks.0.attention.linear_layers.0.weight torch.Size([64, 64])
generator.bert.transformer_blocks.0.attention.linear_layers.0.bias torch.Size([64])
generator.bert.transformer_blocks.0.attention.linear_layers.1.weight torch.Size([64, 64])
generator.bert.transformer_blocks.0.attention.linear_layers.1.bias torch.Size([64])
generator.bert.transformer_blocks.0.attention.linear_layers.2.weight torch.Size([64, 64])
generator.bert.transforme

In [9]:
print(checkpoint_p['state_dict']['critic.bert.embedding.input.weight'])

tensor([[ 0.5004],
        [-2.5577],
        [ 0.5532],
        [-9.0691],
        [ 2.0405],
        [ 1.9576],
        [ 0.0836],
        [ 0.5649],
        [-0.5833],
        [-0.5987],
        [ 1.0329],
        [ 5.5730],
        [ 1.9099],
        [ 4.0439],
        [ 0.7667],
        [ 1.3041],
        [ 0.2591],
        [ 1.5450],
        [ 6.0901],
        [ 2.3652],
        [ 3.0429],
        [-3.6318],
        [ 2.1024],
        [-0.3306],
        [ 0.4442],
        [ 5.3540],
        [ 5.7043],
        [-0.4855],
        [ 2.4442],
        [-0.2827],
        [-0.6130],
        [ 0.6491],
        [-1.9083],
        [ 1.0670],
        [-0.6086],
        [ 0.6371],
        [ 1.2413],
        [-1.5205],
        [ 5.3508],
        [ 0.5478],
        [-3.5981],
        [-1.0268],
        [ 0.4822],
        [ 0.7821],
        [-2.2264],
        [ 1.6432],
        [ 1.1424],
        [ 1.7548],
        [ 0.8087],
        [ 2.2181],
        [ 0.6237],
        [ 0.8127],
        [ 1.

**Run each GAN on some arbitrary input kinematics:**

In [21]:
'''
Proton GAN
'''
import numpy as np

# Set your kinematics here:
# ke = 30.3  # Initial kinetic energy
# ini_dir = [0.9999999999999999, 0.0, 0.0]  # Initial direction
# ini_pos = [-1.5, -4.2, 2.7]  # Initial 3D position (mm)

# get one event from the test set
event = test_set_p[15]
# convert torch tensor to numpy array
ke = float(event['ke'].numpy())
ini_pos = event['pos_ini'].numpy()
ini_dir = event['dir_ini'].numpy()
if args.particle == "proton_exiting" or args.particle == "muon":
    exit_pos = event['pos_exit'].numpy()
else:
    exit_pos = None
img = event['image'].numpy()

#print(ke, ini_pos, ini_dir, exit_pos, img)

if exit_pos is not None:
    params = np.array([ini_pos[0], ini_pos[1], ini_pos[2], ke, ini_dir[0], ini_dir[1], ini_dir[2], exit_pos[0], exit_pos[1], exit_pos[2]])
else:
    params = np.array([ini_pos[0], ini_pos[1], ini_pos[2], ke, ini_dir[0], ini_dir[1], ini_dir[2]])

labels = torch.tensor([params], dtype=torch.float32)

print(labels)




tensor([[ 0.1710,  0.1670, -0.3214,  0.1553, -0.4682, -0.8817,  0.0586]])


/tmp/ipykernel_1129223/1863721312.py:14: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  ke = float(event['ke'].numpy())


In [25]:
with torch.no_grad():
    generated_p = model.sample(labels)

print(generated_p)

tensor([[[-0.9938, -1.0026, -0.9878, -1.0046, -1.0010, -0.9969, -0.9884,
          -0.9909, -1.0024, -0.9982, -1.0023, -0.9975, -0.9994, -0.9971,
          -0.9954, -0.9924, -1.0002, -0.9966, -0.9997, -0.9994, -0.9996,
          -0.9974, -0.9971, -1.0003, -0.9988, -1.0005, -0.9795, -0.9976,
          -0.9997, -0.9983, -1.0020, -0.7355, -0.8515, -1.0020, -1.0006,
          -0.9911, -0.9906, -0.9548, -0.9963, -0.9994, -0.9989, -0.9965,
          -1.0021, -0.9972, -0.9983, -0.9912, -0.9993, -0.9968, -0.9999,
          -1.0039, -0.9961, -1.0133, -0.9909, -1.0011, -1.0012, -0.9935,
          -0.9348, -0.9673, -0.9935, -0.9976, -0.9932, -0.8873, -0.4535,
          -0.9718, -1.0000, -0.9944, -0.9961, -0.9943, -0.9971, -1.0064,
          -0.9989, -0.9993, -0.9997, -0.9984, -1.0030, -0.9967, -1.0000,
          -0.9998, -0.9971, -0.9985, -0.9967, -0.9963, -0.9994, -0.9974,
          -0.9984, -1.0006, -0.9977, -0.9961, -0.9995, -0.9987, -0.9988,
          -0.9967, -0.9945, -1.0025, -1.0005, -0.99

tensor([[[  3.0000,   3.0000,   3.0000,   3.0000,   3.0000],
         [  3.0000,   3.0000,   3.0000,   3.0000,   3.0000],
         [  3.0000,   3.0000,   3.0000,   3.0000,   3.0000],
         [  3.0000,   3.0000,   3.0000,   3.0000,   3.0000],
         [  3.0000,   3.0000,   3.0000,   3.0000,   3.0000]],

        [[  3.0000,   3.0000,   3.0000,  39.9746,   3.0000],
         [  3.0000,   3.0000, 230.2248,   3.0000,   3.0000],
         [  3.0000,   3.0000, 100.7135,   3.0000,   3.0000],
         [  3.0000,   3.0000,   3.0000,   3.0000,   3.0000],
         [  3.0000,   3.0000,   3.0000,   3.0000,   3.0000]],

        [[  3.0000,   3.0000,   3.0000,   3.0000,   3.0000],
         [  3.0000,   3.0000, 228.2390,   3.0000,   3.0000],
         [  3.0000,   3.0000, 187.8737,   3.3160,   3.0000],
         [  3.0000,   3.0000,   3.0000,   3.0000,   3.0000],
         [  3.0000,   3.0000,   3.0000,   3.0000,   3.0000]],

        [[  3.0000,   3.0000,   3.0000,   3.0000,   3.0000],
         [  3.0000

**Visualise the GAN-generated images:**

In [28]:
'''
Plot the generated images!
'''


#generated_p = generated_p.numpy()
# copy the image to a pure numpy array

print(generated_p.shape)
generated_p_1 = generated_p[0][0]

min_charge = 0
max_charge = test_set_p.metadata['statistics']['per_tree'][args.particle]['recon_charge']['max']

generated_p_1 = (generated_p_1 + 1) / 2
generated_p_1 *= (max_charge - min_charge)
generated_p_1 += min_charge

print(generated_p_1.shape)
# reshape the generated image to a 5x5x5 array
generated_p_1 = generated_p_1.reshape(5, 5, 5)



generated_plot = np.zeros((5, 5, 5))

for i in range(generated_plot.shape[0]):
    for j in range(generated_plot.shape[1]):
        for k in range(generated_plot.shape[2]):
            generated_plot[i, j, k] = float(generated_p_1[i, j, k])
print(generated_plot)
# check the type of the elements in the array
print(generated_plot.dtype)

# Max deposited energy in one voxel
max_energy = generated_plot.max()

#generated_plot[generated_plot > 150] = 0
#generated_plot[generated_plot < 4] = 0
print(max_energy)




torch.Size([1, 1, 125])
torch.Size([125])
[[[ 8.25701046e+00 -3.47229147e+00  1.61765671e+01 -6.14594173e+00
   -1.38577974e+00]
  [ 4.12660408e+00  1.54148388e+01  1.21214933e+01 -3.14164996e+00
    2.37865520e+00]
  [-3.00920320e+00  3.34538984e+00  8.11632514e-01  3.86281943e+00
    6.07971859e+00]
  [ 1.00764894e+01 -2.02472448e-01  4.47118759e+00  3.61456096e-01
    7.37250030e-01]
  [ 5.52363217e-01  3.44123936e+00  3.82012272e+00 -3.82606387e-01
    1.60560012e+00]]

 [[-7.17367172e-01  2.72502098e+01  3.22236967e+00  3.37295651e-01
    2.28367686e+00]
  [-2.68204689e+00  3.51474182e+02  1.97419357e+02 -2.70264292e+00
   -7.43666410e-01]
  [ 1.18242798e+01  1.24610863e+01  6.00596161e+01  4.89918375e+00
    8.56864035e-01]
  [ 1.47648048e+00  4.68958187e+00 -2.83271313e+00  3.74241328e+00
    2.19741225e+00]
  [ 1.16524639e+01  9.26810503e-01  4.28115177e+00  1.36565924e-01
   -5.16684961e+00]]

 [[ 5.15290785e+00 -1.76417198e+01  1.20604191e+01 -1.41683185e+00
   -1.58001387e+0

In [14]:
generated_plot = generated_plot.astype(np.float32)

In [29]:

print("- Proton image:")
plotly_generate(generated_plot, max_energy=max_energy)


- Proton image:


In [ ]:
event = test_set_p[15]
true_img = np.zeros((125,))

for i in range(true_img.shape[0]):
    true_img[i] = float(event["image"][i])

true_img = true_img.reshape(5, 5, 5)

# get back the normalization
min_charge = 0
max_charge = test_set_p.metadata['statistics']['per_tree'][args.particle]['recon_charge']['max']
true_img = (true_img + 1) / 2
true_img *= (max_charge - min_charge)
true_img += min_charge

print(test_set_p.metadata['statistics']['per_tree']['proton_contained']['recon_charge']['std'])
print(test_set_p.metadata['statistics']['per_tree']['proton_contained']['recon_charge']['mean'])

max_energy = true_img.max()

print(true_img)

print("- True image:")
plotly_generate(true_img, max_energy=max_energy)



303.3067761656923
197.1132585084357
[[[-8.43769499e-14 -8.43769499e-14 -8.43769499e-14 -8.43769499e-14
   -8.43769499e-14]
  [-8.43769499e-14 -8.43769499e-14 -8.43769499e-14 -8.43769499e-14
   -8.43769499e-14]
  [-8.43769499e-14 -8.43769499e-14 -8.43769499e-14 -8.43769499e-14
   -8.43769499e-14]
  [-8.43769499e-14 -8.43769499e-14 -8.43769499e-14 -8.43769499e-14
   -8.43769499e-14]
  [-8.43769499e-14 -8.43769499e-14 -8.43769499e-14 -8.43769499e-14
   -8.43769499e-14]]

 [[-8.43769499e-14 -8.43769499e-14 -8.43769499e-14  2.35980667e+02
   -8.43769499e-14]
  [-8.43769499e-14 -8.43769499e-14  3.43481979e+01  3.48438988e+01
   -8.43769499e-14]
  [-8.43769499e-14 -8.43769499e-14 -8.43769499e-14 -8.43769499e-14
   -8.43769499e-14]
  [-8.43769499e-14 -8.43769499e-14 -8.43769499e-14 -8.43769499e-14
   -8.43769499e-14]
  [-8.43769499e-14 -8.43769499e-14 -8.43769499e-14 -8.43769499e-14
   -8.43769499e-14]]

 [[-8.43769499e-14 -8.43769499e-14 -8.43769499e-14 -8.43769499e-14
   -8.43769499e-14]
  [

In [16]:
print(event["image"].numpy())

[ 197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  266.32901016  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  261.34530654  197.11325851  197.11325851
  197.11325851  263.76361099 1348.25107589  298.85057845  197.11325851
  197.11325851  197.11325851  286.39696517  197.11325851  197.11325851
  197.